In [61]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, losses 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [62]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# imgs
print(x_train.shape == (60000, 28, 28))
print(x_test.shape == (10000, 28, 28))

# labels 
print(y_train.shape == (60000,))
print(y_test.shape == (10000,))

True
True
True
True


In [63]:
model = models.Sequential([
    layers.Input((28,28,1)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

In [64]:
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'],)
model.fit(x=x_train, y=y_train, batch_size=32, epochs=4, validation_data=(x_test, y_test))

Epoch 1/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9080 - loss: 0.2937 - val_accuracy: 0.9837 - val_loss: 0.0520
Epoch 2/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9634 - loss: 0.1228 - val_accuracy: 0.9876 - val_loss: 0.0354
Epoch 3/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9721 - loss: 0.0941 - val_accuracy: 0.9886 - val_loss: 0.0339
Epoch 4/4
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9774 - loss: 0.0754 - val_accuracy: 0.9925 - val_loss: 0.0255


In [65]:
model_loss, model_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9925 - loss: 0.0255


In [66]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.025456586852669716
Accuracy: 99.25


In [67]:
predictions = model.predict(x_test)
decisions = np.argmax(predictions, axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [68]:
print(classification_report(y_test, decisions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       980
           1       1.00      1.00      1.00      1135
           2       0.99      0.99      0.99      1032
           3       0.99      1.00      0.99      1010
           4       0.99      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       0.99      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [69]:
cm = confusion_matrix(y_test, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,976,0,0,0,0,0,2,1,1,0
1,0,1134,0,0,0,1,0,0,0,0
2,1,1,1019,3,0,0,1,6,1,0
3,0,0,1,1005,0,1,0,2,1,0
4,0,0,0,0,976,0,1,0,1,4
5,0,0,0,5,0,884,1,0,1,1
6,1,2,1,0,1,3,950,0,0,0
7,0,2,3,1,0,0,0,1019,2,1
8,1,0,1,0,1,1,0,2,965,3


In [70]:
model.save('digit-recognizer.keras')